# Sylhet Tea Candidate Verification Pipeline

Self-contained notebook: parses `sylhet_tea_candidates.kml`, pulls high-resolution
satellite chips for each candidate polygon from Esri World Imagery (public tile
endpoint, no API key needed), and runs a quantitative check for planted-row
texture (FFT periodicity + Hough line straightness) vs. natural forest.

**Inputs needed in the same folder:**
- `sylhet_tea_candidates.kml`
- `sylhet_tea_candidates_review.csv` (or the completed version)

**Outputs:**
- `chips/TEA-Cxxx.png` -- one high-res image per candidate
- `sylhet_tea_candidates_review_cv.csv` -- original CSV + `cv_row_score`, `cv_verdict`, `cv_notes`

Note on resolution: Esri's World Imagery basemap resolution varies by region --
most of Sylhet division has sub-meter to ~1m coverage, but pockets can be coarser.
The notebook reports the coverage/zoom it actually got for each chip so you can
tell which results are trustworthy vs. which need a manual look.


In [ ]:
# 1. Install dependencies
!pip install -q contextily rasterio shapely opencv-python-headless scikit-image matplotlib numpy pandas


In [ ]:
# 2. Parse the KML into polygons
import re, json
import numpy as np

KML_PATH = "sylhet_tea_candidates.kml"

def parse_kml(path):
    xml = open(path, encoding="utf-8").read()
    placemarks = re.findall(r"<Placemark>(.*?)</Placemark>", xml, re.S)
    out = {}
    for pm in placemarks:
        name = re.search(r"<name>(.*?)</name>", pm).group(1)
        cid = name.split()[0]
        desc_m = re.search(r"<description><!\[CDATA\[(.*?)\]\]></description>", pm, re.S)
        desc = desc_m.group(1) if desc_m else ""
        upz_m = re.search(r"upazila:\s*([^<]+)<", desc)
        upazila = upz_m.group(1).strip() if upz_m else None
        coord_blocks = re.findall(r"<coordinates>(.*?)</coordinates>", pm, re.S)
        rings = []
        for block in coord_blocks:
            pts = [(float(p.split(",")[0]), float(p.split(",")[1]))
                   for p in block.strip().split()]
            rings.append(pts)
        out[cid] = {"upazila": upazila, "rings": rings}
    return out

candidates = parse_kml(KML_PATH)
print(f"Parsed {len(candidates)} candidates")
for cid, v in list(candidates.items())[:3]:
    print(cid, v["upazila"], f"{len(v['rings'][0])} vertices")


In [ ]:
# 3. Fetch a high-res image chip per candidate from Esri World Imagery
import contextily as cx
import matplotlib.pyplot as plt
import os

os.makedirs("chips", exist_ok=True)

def bbox_of(rings, pad_frac=0.15):
    xs = [p[0] for ring in rings for p in ring]
    ys = [p[1] for ring in rings for p in ring]
    minx, maxx = min(xs), max(xs)
    miny, maxy = min(ys), max(ys)
    padx = (maxx - minx) * pad_frac or 0.001
    pady = (maxy - miny) * pad_frac or 0.001
    return minx - padx, miny - pady, maxx + padx, maxy + pady

ESRI = ("https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}")

chip_meta = {}
for cid, v in candidates.items():
    minx, miny, maxx, maxy = bbox_of(v["rings"])
    try:
        img, ext = cx.bounds2img(minx, miny, maxx, maxy, zoom=18,
                                  source=ESRI, ll=True)
        out_path = f"chips/{cid}.png"
        plt.imsave(out_path, img)
        chip_meta[cid] = {"path": out_path, "extent": ext, "zoom": 18,
                           "shape": img.shape[:2]}
        print(f"{cid}: OK  {img.shape[1]}x{img.shape[0]} px")
    except Exception as e:
        chip_meta[cid] = {"error": str(e)}
        print(f"{cid}: FAILED -- {e}")


## 4. Quantitative row-pattern check

Two independent signals, combined into a `cv_row_score` (0-1, higher = more
plantation-like):

- **FFT periodicity** -- a real tea block has a dominant spatial frequency
  (row spacing); natural forest canopy doesn't. We take a 2D FFT of the
  grayscale chip and measure how concentrated the energy is in a ring of
  frequencies vs. spread uniformly.
- **Hough line straightness** -- plantation service tracks and row edges show
  up as long straight lines; ragged natural-forest edges don't. We run Canny
  edge detection + probabilistic Hough transform and count long straight
  segments per unit area.

This is a heuristic, not ground truth -- always spot-check the borderline
cases (`cv_verdict == "uncertain"`) visually.


In [ ]:
import cv2
import numpy as np

def fft_periodicity_score(gray):
    f = np.fft.fft2(gray)
    fshift = np.abs(np.fft.fftshift(f))
    h, w = fshift.shape
    cy, cx_ = h // 2, w // 2
    fshift[cy-2:cy+3, cx_-2:cx_+3] = 0  # zero out DC component
    # ring 5-40px radius (mid frequencies where row spacing typically falls)
    Y, X = np.ogrid[:h, :w]
    r = np.sqrt((Y - cy)**2 + (X - cx_)**2)
    ring_mask = (r > 5) & (r < 40)
    ring_vals = fshift[ring_mask]
    peakiness = (ring_vals.max() / (ring_vals.mean() + 1e-9)) if ring_vals.size else 0
    return float(np.clip(peakiness / 25, 0, 1))  # normalized, tune threshold after eyeballing a few chips

def hough_line_score(gray):
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40,
                             minLineLength=gray.shape[0]//6, maxLineGap=10)
    if lines is None:
        return 0.0
    n_long_lines = len(lines)
    area = gray.shape[0] * gray.shape[1]
    density = n_long_lines / (area / 1e5)  # lines per 100k px
    return float(np.clip(density / 8, 0, 1))  # normalized, tune after eyeballing

results = {}
for cid, meta in chip_meta.items():
    if "error" in meta:
        results[cid] = {"cv_row_score": None, "cv_verdict": "no_image",
                         "cv_notes": meta["error"]}
        continue
    img = cv2.imread(meta["path"])
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    fft_s = fft_periodicity_score(gray)
    hough_s = hough_line_score(gray)
    combined = round(0.5 * fft_s + 0.5 * hough_s, 3)
    if combined >= 0.6:
        verdict = "likely tea (regular pattern detected)"
    elif combined <= 0.3:
        verdict = "likely not tea (no regular pattern)"
    else:
        verdict = "uncertain -- needs visual check"
    results[cid] = {"cv_row_score": combined, "cv_verdict": verdict,
                     "cv_notes": f"fft={fft_s:.2f}, hough={hough_s:.2f}, zoom={meta['zoom']}"}
    print(cid, combined, verdict)


In [ ]:
# 5. Visual sanity-check grid -- ALWAYS look at these yourself before trusting the scores
import math

ids = sorted(chip_meta.keys())
n = len(ids)
cols = 4
rows_n = math.ceil(n / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols*3.2, rows_n*3.2))
axes = axes.flatten()
for ax, cid in zip(axes, ids):
    meta = chip_meta[cid]
    if "error" in meta:
        ax.set_title(f"{cid}\n(no image)", fontsize=8)
        ax.axis("off")
        continue
    img = plt.imread(meta["path"])
    ax.imshow(img)
    r = results[cid]
    ax.set_title(f"{cid}\nscore={r['cv_row_score']}", fontsize=8)
    ax.axis("off")
for ax in axes[len(ids):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig("chips/_overview_grid.png", dpi=120)
plt.show()


In [ ]:
# 6. Merge results back into the review CSV
import pandas as pd

df = pd.read_csv("sylhet_tea_candidates_review.csv")
df["cv_row_score"] = df["candidate_id"].map(lambda c: results.get(c, {}).get("cv_row_score"))
df["cv_verdict"]   = df["candidate_id"].map(lambda c: results.get(c, {}).get("cv_verdict"))
df["cv_notes"]     = df["candidate_id"].map(lambda c: results.get(c, {}).get("cv_notes"))

out_path = "sylhet_tea_candidates_review_cv.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
df


## Notes / caveats

- **Thresholds in `fft_periodicity_score` / `hough_line_score` are unturned.**
  Look at `chips/_overview_grid.png` next to the printed scores for 4-5 known
  cases (e.g. TEA-C015 = confirmed Loobacherra) and adjust the normalization
  constants (`/25`, `/8`) until scores line up with what you see.
- Esri World Imagery resolution isn't guaranteed uniform -- check the `zoom`
  and pixel dimensions printed in step 3; a chip that came back tiny/blurry
  should be treated as `no_image`, not trusted.
- This automates the *first pass* of the KML's own instructions ("zoom in,
  look for row patterns"). Treat `uncertain` and any borderline `likely`
  results as needing your own eyes on the grid image, not final answers.
